# G1 mode switching

This notebook builds a small control panel for switching the Unitree G1 between the main locomotion modes used during the academy. Run the cells from top to bottom while the robot is powered, connected to the same DDS network interface, and in a safe open area.

Mode switching sends real commands to the robot. Keep one person responsible for the emergency stop and only press buttons when the robot state shown by the panel matches what you expect.


The first code cell imports the Unitree SDK classes used by this notebook. `LocoClient` sends locomotion finite-state-machine commands, while `MotionSwitcherClient` checks or releases the higher-level motion service that can block direct locomotion control.


In [3]:
try:
    from unitree_sdk2py.core.channel import ChannelFactoryInitialize
    from unitree_sdk2py.comm.motion_switcher.motion_switcher_client import MotionSwitcherClient
    from unitree_sdk2py.g1.loco.g1_loco_api import (
        ROBOT_API_ID_LOCO_GET_FSM_ID,
        ROBOT_API_ID_LOCO_GET_FSM_MODE,
    )
    from unitree_sdk2py.g1.loco.g1_loco_client import LocoClient
except ImportError as exc:
    raise SystemExit(
        "unitree_sdk2py is not installed. Install it with:\n"
        "  pip install -e <path-to-unitree_sdk2_python>"
    ) from exc


This cell imports standard Python helpers and notebook UI libraries. The lock protects SDK calls when several UI callbacks happen close together, and `ipywidgets` provides the buttons, status text, and details panel inside Jupyter.


In [4]:
import json
import os
import threading
import time
from dataclasses import dataclass
from typing import Any

import ipywidgets as widgets
from IPython.display import display


These constants give readable names to the finite-state-machine IDs used by the G1 locomotion service. The error hints translate common motion-switcher return codes into messages that are easier to interpret during an exercise.


In [5]:
FSM_ZERO_TORQUE = 0
FSM_DAMPING = 1
FSM_SIT = 3
FSM_PREPARE = 4
FSM_WALK = 501
FSM_RUN = 802
DEFAULT_CLIMB_FSM = int(os.environ.get("G1_CLIMB_FSM_ID", "812"))
AI_MODE_NAME = "ai_sport"
MODE_ALIASES = {
    "ai": AI_MODE_NAME,
}
ERROR_HINTS = {
    0: "success",
    7001: "request parameter error",
    7002: "service busy; retry",
    7004: "unsupported mode name",
    7005: "internal command execute error",
    7006: "check command execute error",
    7007: "switch command execute error",
    7008: "release command execute error",
    7009: "custom config set error",
}


`RobotState` is a small structured snapshot of what the robot reports. Keeping the state in one object makes it clear which values came from locomotion RPCs, which came from the motion switcher, and whether any part of the probe failed.


In [6]:
@dataclass(frozen=True)
class RobotState:
    mode: str
    fsm_id: int | None
    fsm_mode: int | None
    motion_mode: str | None
    motion_raw: Any
    motion_code: int | None = None
    loco_skipped: bool = False
    error: str | None = None


def error_hint(code: int | None) -> str:
    if code is None:
        return ""
    hint = ERROR_HINTS.get(int(code))
    return "" if hint is None else f" ({hint})"


def state_text(state: RobotState) -> str:
    bits = [f"state={state.mode}"]
    if state.fsm_id is not None:
        bits.append(f"fsm_id={state.fsm_id}")
    if state.fsm_mode is not None:
        bits.append(f"fsm_mode={state.fsm_mode}")
    if state.motion_mode:
        bits.append(f"motion={state.motion_mode}")
    if state.motion_code is not None:
        bits.append(f"motion_code={state.motion_code}{error_hint(state.motion_code)}")
    if state.loco_skipped:
        bits.append("loco_rpc=skipped")
    if state.error:
        bits.append(f"error={state.error}")
    return "  ".join(bits)


def state_detail(state: RobotState) -> str:
    data = {
        "mode": state.mode,
        "fsm_id": state.fsm_id,
        "fsm_mode": state.fsm_mode,
        "motion_mode": state.motion_mode,
        "motion_code": state.motion_code,
        "motion_hint": ERROR_HINTS.get(state.motion_code) if state.motion_code is not None else None,
        "loco_rpc": "skipped while motion switcher is released/dev" if state.loco_skipped else "active",
        "motion_raw": state.motion_raw,
        "error": state.error,
    }
    return json.dumps(data, default=str, indent=2, sort_keys=True)


The `Robot` class hides the raw SDK calls behind academy-friendly methods. It initializes the DDS channel lazily, reads the current mode, classifies FSM IDs into names, and exposes one method per button in the UI.


In [7]:
class Robot:
    def __init__(self, iface="eth0", domain_id=0, timeout=10.0, climb_fsm_id=DEFAULT_CLIMB_FSM):
        self.iface = str(iface)
        self.domain_id = int(domain_id)
        self.timeout = float(timeout)
        self.climb_fsm_id = int(climb_fsm_id)
        self._lock = threading.RLock()
        self._initialized = False
        self._loco = None
        self._motion = None

    def _ensure_clients(self):
        if self._initialized:
            return
        ChannelFactoryInitialize(self.domain_id, self.iface)

        loco = LocoClient()
        loco.SetTimeout(self.timeout)
        loco.Init()

        motion = MotionSwitcherClient()
        motion.SetTimeout(self.timeout)
        motion.Init()

        self._loco = loco
        self._motion = motion
        self._initialized = True

    @staticmethod
    def _result_code(result):
        if result is None:
            return 0
        if isinstance(result, tuple):
            return int(result[0])
        return int(result)

    @staticmethod
    def _rpc_get_int(client, api_id):
        try:
            code, data = client._Call(api_id, "{}")
            if code != 0 or not data:
                return None
            return int(json.loads(data).get("data"))
        except Exception:
            return None

    @staticmethod
    def _motion_mode_name(data):
        if not isinstance(data, dict):
            return None
        for key in ("name", "mode", "alias"):
            value = data.get(key)
            if isinstance(value, str) and value.strip():
                return value.strip()
        return None

    @staticmethod
    def _canonical_motion_name(name):
        value = "" if name is None else str(name).strip()
        return MODE_ALIASES.get(value, value)

    @classmethod
    def _motion_is_ai(cls, name):
        return cls._canonical_motion_name(name) == AI_MODE_NAME

    def _classify_mode(self, fsm_id):
        if fsm_id == FSM_ZERO_TORQUE:
            return "zero_torque"
        if fsm_id == FSM_DAMPING:
            return "damping"
        if fsm_id == FSM_SIT:
            return "sit"
        if fsm_id == FSM_PREPARE:
            return "prepare"
        if fsm_id == FSM_WALK:
            return "walk"
        if fsm_id == FSM_RUN:
            return "run"
        if fsm_id == self.climb_fsm_id:
            return "climb"
        return "unknown"

    def get_mode(self):
        with self._lock:
            try:
                self._ensure_clients()

                motion_raw = None
                motion_name = None
                motion_code = None
                try:
                    motion_code, motion_raw = self._motion.CheckMode()
                    motion_code = int(motion_code)
                    if motion_code == 0:
                        motion_name = self._motion_mode_name(motion_raw)
                except Exception as exc:
                    motion_raw = {"error": str(exc)}

                if motion_code == 0 and not self._motion_is_ai(motion_name):
                    return RobotState(
                        mode="dev",
                        fsm_id=None,
                        fsm_mode=None,
                        motion_mode=motion_name or "<released>",
                        motion_raw=motion_raw,
                        motion_code=motion_code,
                        loco_skipped=True,
                    )

                fsm_id = self._rpc_get_int(self._loco, ROBOT_API_ID_LOCO_GET_FSM_ID)
                fsm_mode = self._rpc_get_int(self._loco, ROBOT_API_ID_LOCO_GET_FSM_MODE)
                return RobotState(
                    mode=self._classify_mode(fsm_id),
                    fsm_id=fsm_id,
                    fsm_mode=fsm_mode,
                    motion_mode=motion_name,
                    motion_raw=motion_raw,
                    motion_code=motion_code,
                )
            except Exception as exc:
                return RobotState("unavailable", None, None, None, None, error=str(exc))

    def switch_mode_damping(self):
        with self._lock:
            self._ensure_clients()
            code = self._result_code(self._loco.Damp())
            return f"Damping command sent. code={code}{error_hint(code)}"

    def switch_mode_zero_torque(self):
        with self._lock:
            self._ensure_clients()
            state = self.get_mode()
            if state.mode == "dev":
                code = self._result_code(self._motion.SelectMode(AI_MODE_NAME))
                time.sleep(0.2)
                try:
                    self._loco.ZeroTorque()
                except Exception:
                    pass
                return f"AI mode selected for zero torque. code={code}{error_hint(code)}"
            code = self._result_code(self._loco.ZeroTorque())
            return f"Zero torque command sent. code={code}{error_hint(code)}"

    def switch_mode_prepare(self):
        with self._lock:
            self._ensure_clients()
            code = self._result_code(self._loco.SetFsmId(FSM_PREPARE))
            return f"Prepare command sent. fsm_id={FSM_PREPARE} code={code}{error_hint(code)}"

    def switch_mode_sit(self):
        with self._lock:
            self._ensure_clients()
            code = self._result_code(self._loco.Sit())
            return f"Sit command sent. code={code}{error_hint(code)}"

    def switch_mode_walk(self):
        with self._lock:
            self._ensure_clients()
            code = self._result_code(self._loco.SetFsmId(FSM_WALK))
            return f"Walk command sent. fsm_id={FSM_WALK} code={code}{error_hint(code)}"

    def switch_mode_run(self):
        with self._lock:
            self._ensure_clients()
            code = self._result_code(self._loco.SetFsmId(FSM_RUN))
            return f"Run command sent. fsm_id={FSM_RUN} code={code}{error_hint(code)}"

    def switch_mode_climb(self):
        with self._lock:
            self._ensure_clients()
            code = self._result_code(self._loco.SetFsmId(self.climb_fsm_id))
            return f"Climb command sent. fsm_id={self.climb_fsm_id} code={code}{error_hint(code)}"

    def switch_mode_dev(self):
        with self._lock:
            self._ensure_clients()
            state = self.get_mode()
            if state.mode == "dev":
                code = self._result_code(self._motion.SelectMode(AI_MODE_NAME))
                return f"AI mode selected; dev mode off. code={code}{error_hint(code)}"
            code = self._result_code(self._motion.ReleaseMode())
            return f"AI mode released; dev mode active. code={code}{error_hint(code)}"

    def command(self, name):
        methods = {
            "damping": self.switch_mode_damping,
            "zero_torque": self.switch_mode_zero_torque,
            "prepare": self.switch_mode_prepare,
            "sit": self.switch_mode_sit,
            "walk": self.switch_mode_walk,
            "run": self.switch_mode_run,
            "climb": self.switch_mode_climb,
            "dev": self.switch_mode_dev,
        }
        if name not in methods:
            raise ValueError(f"Unknown command: {name}")
        return methods[name]()


The next helpers encode the same button availability rules used by the standalone mode-control script. They are intentionally conservative: for example, walk/run/climb are enabled only when the robot is already in a locomotion-ready family of modes.


In [8]:
def button_disabled(mode, button):
    if mode == "unavailable":
        return True
    if mode == "zero_torque":
        return button not in {"damping", "dev"}
    if mode == "dev":
        return button != "dev"
    if button in {"walk", "run", "climb"}:
        return mode not in {"prepare", "walk", "run", "climb"}
    return False


Create the robot connection object here. Change `IFACE` if your robot network is not on `eth0`, and change `DOMAIN_ID` only if your academy setup uses a non-default DDS domain.


In [9]:
IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
TIMEOUT = 10.0

robot = Robot(iface=IFACE, domain_id=DOMAIN_ID, timeout=TIMEOUT)
print(f"Robot client configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Robot client configured for iface='eth0', domain_id=0.


Run this cell whenever you want a quick text-only mode check. It is useful before opening the full control panel, because it verifies that DDS discovery and the SDK clients can talk to the robot.


In [10]:
current_state = robot.get_mode()
print(state_text(current_state))
print(state_detail(current_state))


[ChannelFactory] create domain error. msg: Occurred upon initialisation of a cyclonedds.domain.Domain
state=unavailable  error=channel factory init error.
{
  "error": "channel factory init error.",
  "fsm_id": null,
  "fsm_mode": null,
  "loco_rpc": "active",
  "mode": "unavailable",
  "motion_code": null,
  "motion_hint": null,
  "motion_mode": null,
  "motion_raw": null
}


1780691320.256504 [0]    python3: eth0: does not match an available interface.


This final cell creates the notebook control panel. The buttons call the `Robot` methods above, refresh the state display after every command, and keep a short event log so participants can see what was sent during the exercise.


In [11]:
button_specs = [
    ("damping", "Damping", "warning"),
    ("zero_torque", "Zero Torque", "danger"),
    ("prepare", "Prepare", "primary"),
    ("sit", "Sit", "secondary"),
    ("walk", "Walk", "success"),
    ("run", "Run", "success"),
    ("climb", "Climb", "success"),
    ("dev", "Dev Off", "secondary"),
]

status_label = widgets.HTML(value="")
command_status = widgets.HTML(value="")
state_box = widgets.Textarea(
    value="",
    layout=widgets.Layout(width="100%", height="220px"),
    disabled=True,
)
event_log_box = widgets.Textarea(
    value="",
    layout=widgets.Layout(width="100%", height="120px"),
    disabled=True,
)
refresh_button = widgets.Button(description="Refresh", button_style="info")
mode_buttons = {}
event_log = []


def _set_button_style(button, style_name):
    # ipywidgets supports a limited set of Bootstrap-style names.
    if style_name in {"primary", "success", "info", "warning", "danger"}:
        button.button_style = style_name
    else:
        button.button_style = ""


for name, label, style_name in button_specs:
    button = widgets.Button(
        description=label,
        layout=widgets.Layout(width="150px", height="44px"),
    )
    _set_button_style(button, style_name)
    mode_buttons[name] = button


def refresh_panel(status=None):
    state = robot.get_mode()
    status_label.value = f"<b>{state_text(state)}</b>"
    state_box.value = state_detail(state)

    for name, button in mode_buttons.items():
        button.disabled = button_disabled(state.mode, name)

    dev_button = mode_buttons["dev"]
    if state.mode == "dev":
        dev_button.description = "Dev Off"
        _set_button_style(dev_button, "")
    else:
        dev_button.description = "Dev On"
        _set_button_style(dev_button, "secondary")

    if status is not None:
        command_status.value = status
    return state


def on_command(name):
    def _handler(_button):
        try:
            before = robot.get_mode()
            if button_disabled(before.mode, name):
                message = f"{name.replace('_', ' ').title()} is disabled from {before.mode}."
            else:
                message = robot.command(name)
        except Exception as exc:
            message = f"Command failed: {exc}"

        event_log.insert(0, f"{time.strftime('%H:%M:%S')} {name}: {message}")
        del event_log[20:]
        event_log_box.value = "\n".join(event_log)
        refresh_panel(message)
    return _handler


for name, button in mode_buttons.items():
    button.on_click(on_command(name))

refresh_button.on_click(lambda _button: refresh_panel("State refreshed."))

button_rows = [
    widgets.HBox([mode_buttons["damping"], mode_buttons["zero_torque"], mode_buttons["prepare"], mode_buttons["sit"]]),
    widgets.HBox([mode_buttons["walk"], mode_buttons["run"], mode_buttons["climb"], mode_buttons["dev"], refresh_button]),
]

refresh_panel("Panel ready.")
display(widgets.VBox([status_label, *button_rows, command_status, state_box, event_log_box]))


[ChannelFactory] create domain error. msg: Occurred upon initialisation of a cyclonedds.domain.Domain


1780691324.348101 [0]    python3: eth0: does not match an available interface.
